<a href="https://colab.research.google.com/github/AntonRvn/SlovoDataset/blob/main/Slovo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Загрузка библиотек**

In [1]:
!pip install -q ijson pyarrow

Импорт библиотек

In [2]:
from google.colab import drive

import ijson
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path
from tqdm import tqdm

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split, StratifiedKFold

[Текст ссылки](https://)**Подключение к Google Drive**

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Пути к файлам

In [4]:
JSON_PATH = Path(
    "/content/drive/MyDrive/slovo_mediapipe.json"
)

ANNOTATIONS_PATH = Path(
    "/content/drive/MyDrive/annotations.csv"
)

Загрузка файла аннотаций

In [5]:
annotations = pd.read_csv(
    ANNOTATIONS_PATH,
    sep='\t'
)

annotations['attachment_id'] = (
    annotations['attachment_id']
    .astype(str)
)

print(annotations.shape)

annotations.head()

(20400, 9)


,attachment_id,text,user_id,height,width,length,train,begin,end
0,44e8d2a0-7e01-450b-90b0-beb7400d2c1e,Ё,185bd3a81d9d618518d10abebf0d17a8,1920,1080,156.0,True,36,112
1,df5b08f0-41d1-4572-889c-8b893e71069b,А,185bd3a81d9d618518d10abebf0d17a8,1920,1080,150.0,True,36,76
2,17f53df4-c467-4aff-9f48-20687b63d49a,Р,185bd3a81d9d618518d10abebf0d17a8,1920,1080,133.0,True,40,97
3,e3add916-c708-4339-ad98-7e2740be29e9,Е,185bd3a81d9d618518d10abebf0d17a8,1920,1080,144.0,True,43,107
4,bd7272ed-1850-48f1-a2a8-c8fed523dc37,Ч,185bd3a81d9d618518d10abebf0d17a8,1920,1080,96.0,True,20,70


Кодирование классов жестов

In [6]:
label_encoder = LabelEncoder()

annotations['label'] = label_encoder.fit_transform(annotations['text'])

num_classes = annotations['label'].nunique()

print("Количество классов:", num_classes)

Количество классов: 1001


*Создание* словарей labels и train/test split

In [7]:
label_map = dict(
    zip(
        annotations['attachment_id'],
        annotations['label']
    )
)

train_map = dict(
    zip(
        annotations['attachment_id'],
        annotations['train']
    )
)

Формирование sequence dataset из JSON

In [8]:
MAX_LEN = 64

sequences = []
labels = []
attachment_ids_list = []

invalid_hands = set()
invalid_points = 0

print("Начинаю обработку JSON...")

with open(JSON_PATH, 'rb') as f:

    parser = ijson.kvitems(f, '')

    for attachment_id, frames in tqdm(parser):

        attachment_id = str(attachment_id)

        if attachment_id not in label_map:
            continue

        if not isinstance(frames, list):
            continue

        sequence = []

        for frame in frames:

            if not isinstance(frame, dict):
                continue

            frame_vector = []

            for hand_name in ['hand 1', 'hand 2']:

                landmarks = frame.get(hand_name)

                if landmarks is None or not isinstance(landmarks, list):
                    frame_vector.extend([0.0] * 63)
                    continue

                if len(landmarks) != 21:
                    invalid_points += 1

                coords = []

                for point in landmarks[:21]:

                    try:
                        coords.append([
                            float(point['x']),
                            float(point['y']),
                            float(point['z'])
                        ])
                    except Exception:
                        coords.append([0.0, 0.0, 0.0])

                coords = np.array(coords, dtype=np.float32)

                wrist = coords[0]
                coords = coords - wrist

                distances = np.linalg.norm(
                    coords,
                    axis=1,
                    keepdims=True
                )

                coords = np.concatenate(
                    [coords, distances],
                    axis=1
                )

                coords = coords.flatten().tolist()

                coords = coords[:63]
                coords += [0.0] * (63 - len(coords))

                frame_vector.extend(coords)

            sequence.append(frame_vector)

        if len(sequence) == 0:
            continue

        sequence = np.array(sequence, dtype=np.float32)

        if len(sequence) >= MAX_LEN:
            sequence = sequence[:MAX_LEN]
        else:
            pad = np.zeros((MAX_LEN - len(sequence), 126))
            sequence = np.vstack([sequence, pad])

        sequences.append(sequence)
        labels.append(label_map[attachment_id])
        attachment_ids_list.append(attachment_id)

print("Обработка завершена")
print(len(sequences))

Начинаю обработку JSON...


20000it [02:53, 115.48it/s]

Обработка завершена
19990


Преобразование в numpy arrays

In [9]:
X = np.array(sequences, dtype=np.float32)
y = np.array(labels)

print(X.shape)
print(y.shape)

(19990, 64, 126)
(19990,)


In [10]:
# Velocity features
# Для модели 2 (и возможно следующих)
velocity_X = np.diff(
    X,
    axis=1,
    prepend=X[:, 0:1, :]
)

acceleration_X = np.diff(
    velocity_X,
    axis=1,
    prepend=velocity_X[:, 0:1, :]
)

X = np.concatenate(
    [
        X,
        velocity_X,
        acceleration_X
    ],
    axis=2
)

Нормализация данных

In [11]:
'''
X_mean = X.mean(axis=(0, 1), keepdims=True)
X_std = X.std(axis=(0, 1), keepdims=True)

X = (X - X_mean) / (X_std + 1e-6)
'''

'\nX_mean = X.mean(axis=(0, 1), keepdims=True)\nX_std = X.std(axis=(0, 1), keepdims=True)\n\nX = (X - X_mean) / (X_std + 1e-6)\n'

split train/val/test + normalization

In [12]:
X_train_val, X_test, y_train_val, y_test, ids_train_val, ids_test = train_test_split(
    X,
    y,
    attachment_ids_list,
    test_size=0.1,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val, ids_train, ids_val = train_test_split(
    X_train_val,
    y_train_val,
    ids_train_val,
    test_size=0.1,
    random_state=42,
    stratify=y_train_val
)

# normalization ONLY from train

X_mean = X_train.mean(
    axis=(0, 1),
    keepdims=True
)

X_std = X_train.std(
    axis=(0, 1),
    keepdims=True
)

X_train = (
    X_train - X_mean
) / (X_std + 1e-6)

X_val = (
    X_val - X_mean
) / (X_std + 1e-6)

X_test = (
    X_test - X_mean
) / (X_std + 1e-6)

Создание Dataset

In [13]:
class GestureDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            y,
            dtype=torch.long
        )

        # sequence lengths
        self.lengths = torch.tensor(
            (X.sum(axis=2) != 0).sum(axis=1),
            dtype=torch.long
        )
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx],
            self.lengths[idx]
        )

In [14]:
train_dataset = GestureDataset(
    X_train,
    y_train
)

val_dataset = GestureDataset(
    X_val,
    y_val
)

test_dataset = GestureDataset(
    X_test,
    y_test
)

Создание DataLoader

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    num_workers=2,
    pin_memory=True
)

Создание baseline LSTM-модели

In [16]:
'''
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=126,
        hidden_size=128,
        num_layers=1,
        num_classes=1000
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]
        hidden = self.dropout(hidden)

        logits = self.fc(hidden)

        return logits
'''

'\nclass LSTMClassifier(nn.Module):\n\n    def __init__(\n        self,\n        input_size=126,\n        hidden_size=128,\n        num_layers=1,\n        num_classes=1000\n    ):\n\n        super().__init__()\n\n        self.lstm = nn.LSTM(\n            input_size=input_size,\n            hidden_size=hidden_size,\n            num_layers=num_layers,\n            batch_first=True\n        )\n\n        self.dropout = nn.Dropout(0.3)\n\n        self.fc = nn.Linear(hidden_size, num_classes)\n\n    def forward(self, x):\n\n        output, (hidden, cell) = self.lstm(x)\n\n        hidden = hidden[-1]\n        hidden = self.dropout(hidden)\n\n        logits = self.fc(hidden)\n\n        return logits\n'

In [17]:
# Модель 3
'''
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=252,
        hidden_size=128,
        num_classes=1000,
        dropout=0.3
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        pooled = output.mean(dim=1)

        pooled = self.dropout(pooled)

        logits = self.fc(pooled)

        return logits
'''

'\nclass LSTMClassifier(nn.Module):\n\n    def __init__(\n        self,\n        input_size=252,\n        hidden_size=128,\n        num_classes=1000,\n        dropout=0.3\n    ):\n\n        super().__init__()\n\n        self.lstm = nn.LSTM(\n            input_size=input_size,\n            hidden_size=hidden_size,\n            batch_first=True,\n            bidirectional=True\n        )\n\n        self.dropout = nn.Dropout(dropout)\n\n        self.fc = nn.Linear(\n            hidden_size * 2,\n            num_classes\n        )\n\n    def forward(self, x):\n\n        output, (hidden, cell) = self.lstm(x)\n\n        pooled = output.mean(dim=1)\n\n        pooled = self.dropout(pooled)\n\n        logits = self.fc(pooled)\n\n        return logits\n'

In [18]:
class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=378,
        hidden_size=128,
        num_layers=2,
        num_classes=1000,
        dropout=0.3
    ):

        super().__init__()

        # CNN frontend
        self.conv1 = nn.Conv1d(
            input_size,
            256,
            kernel_size=3,
            padding=1
        )

        self.conv_bn = nn.BatchNorm1d(256)

        # LSTM
        self.lstm = nn.LSTM(
            input_size=256,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.batch_norm = nn.BatchNorm1d(hidden_size * 2)
        self.dropout = nn.Dropout(dropout)

        self.fc1 = nn.Linear(hidden_size * 2, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)

        # attention
        self.attention = nn.Sequential(
            nn.Linear(hidden_size * 2, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

    def forward(self, x, lengths):

        # Conv1D
        x = x.transpose(1, 2)
        x = self.conv1(x)
        x = self.conv_bn(x)
        x = torch.relu(x)
        x = x.transpose(1, 2)

        # pack
        packed = nn.utils.rnn.pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True
        )

        # attention logits
        attn = self.attention(output).squeeze(-1)

        # MASK (ИЗМЕНЕНО — КРИТИЧНО)
        mask = torch.arange(output.size(1), device=output.device)[None, :] < lengths[:, None]
        attn = attn.masked_fill(~mask, -1e9)

        attn = torch.softmax(attn, dim=1)

        pooled = torch.sum(
            output * attn.unsqueeze(-1),
            dim=1
        )

        pooled = self.batch_norm(pooled)
        pooled = self.dropout(pooled)

        x = self.fc1(pooled)
        x = self.relu(x)
        x = self.dropout(x)

        logits = self.fc2(x)

        return logits

Инициализация модели

In [19]:
'''
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_classes = len(label_encoder.classes_)

model = LSTMClassifier(
    num_classes=num_classes
).to(device)
'''

device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)

model = LSTMClassifier(
    input_size=378,
    hidden_size=128,
    num_classes=num_classes
).to(device)

print(device)

cuda


Функция потерь и оптимизатор

In [20]:
'''
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)
'''

criterion = nn.CrossEntropyLoss(
    label_smoothing=0.1
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=5
)

Обучение модели

In [21]:
'''
EPOCHS = 50

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)

        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()

    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            X_batch = X_batch.to(device)

            logits = model(X_batch)

            preds = logits.argmax(dim=1)

            val_preds.extend(preds.cpu().numpy())
            val_targets.extend(y_batch.numpy())

    val_acc = accuracy_score(val_targets, val_preds)

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )
'''
EPOCHS = 100

best_val_acc = 0

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for X_batch, y_batch, lengths in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()

        logits = model(
            X_batch,
            lengths
        )

        loss = criterion(
            logits,
            y_batch
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()

    val_preds = []
    val_targets = []

    with torch.no_grad():

        for X_batch, y_batch, lengths in val_loader:

            X_batch = X_batch.to(device)
            lengths = lengths.to(device)

            logits = model(
                X_batch,
                lengths
            )

            preds = logits.argmax(dim=1)

            val_preds.extend(
                preds.cpu().numpy()
            )

            val_targets.extend(
                y_batch.numpy()
            )

    val_acc = accuracy_score(
        val_targets,
        val_preds
    )

    scheduler.step(val_acc)


    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            'best_model.pt'
        )

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {avg_loss:.4f} | "
        f"Val Acc: {val_acc:.4f}"
    )

Epoch 1 | Loss: 6.4572 | Val Acc: 0.0267
Epoch 2 | Loss: 5.5373 | Val Acc: 0.0756
Epoch 3 | Loss: 4.9199 | Val Acc: 0.1544
Epoch 4 | Loss: 4.4607 | Val Acc: 0.2033
Epoch 5 | Loss: 4.1322 | Val Acc: 0.2689
Epoch 6 | Loss: 3.8558 | Val Acc: 0.2900
Epoch 7 | Loss: 3.6434 | Val Acc: 0.3272
Epoch 8 | Loss: 3.4430 | Val Acc: 0.3700
Epoch 9 | Loss: 3.2769 | Val Acc: 0.4217
Epoch 10 | Loss: 3.1450 | Val Acc: 0.4272
Epoch 11 | Loss: 3.0176 | Val Acc: 0.4389
Epoch 12 | Loss: 2.9013 | Val Acc: 0.4706
Epoch 13 | Loss: 2.8133 | Val Acc: 0.4761
Epoch 14 | Loss: 2.7310 | Val Acc: 0.5122
Epoch 15 | Loss: 2.6550 | Val Acc: 0.4944
Epoch 16 | Loss: 2.6080 | Val Acc: 0.5156
Epoch 17 | Loss: 2.5540 | Val Acc: 0.5367
Epoch 18 | Loss: 2.4929 | Val Acc: 0.5339
Epoch 19 | Loss: 2.4517 | Val Acc: 0.5422
Epoch 20 | Loss: 2.4061 | Val Acc: 0.5506
Epoch 21 | Loss: 2.3545 | Val Acc: 0.5567
Epoch 22 | Loss: 2.3429 | Val Acc: 0.5617
Epoch 23 | Loss: 2.2882 | Val Acc: 0.5456
Epoch 24 | Loss: 2.2666 | Val Acc: 0.5672
E

Оценка качества модели

In [22]:
model.eval()

preds = []
targets = []

with torch.no_grad():

    for X_batch, y_batch, lengths in test_loader:

        X_batch = X_batch.to(device)
        lengths = lengths.to(device)

        logits = model(
            X_batch,
            lengths
        )

        pred = logits.argmax(dim=1)

        preds.extend(
            pred.cpu().numpy()
        )

        targets.extend(
            y_batch.numpy()
        )

acc = accuracy_score(
    targets,
    preds
)

print("Accuracy:", acc)

Accuracy: 0.6428214107053527
